<a href="https://colab.research.google.com/github/NeonsCandy/HW16.ipynb/blob/main/HW16.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import tensorflow as tf
from tensorflow.keras.datasets import fashion_mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

(x_train, y_train), (x_test, y_test) = fashion_mnist.load_data()
x_train, x_test = x_train / 255.0, x_test / 255.0

x_train = x_train.reshape(-1, 28, 28, 1)
x_test = x_test.reshape(-1, 28, 28, 1)

model_cnn = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)),
    MaxPooling2D((2, 2)),

    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),

    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(10, activation='softmax')
])

model_cnn.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
history_cnn = model_cnn.fit(x_train, y_train, epochs=15, batch_size=64, validation_split=0.2)

test_loss_cnn, test_acc_cnn = model_cnn.evaluate(x_test, y_test)
print(f"Точність власної CNN: {test_acc_cnn:.4f}")

29515/29515 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
26421880/26421880 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
5148/5148 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
4422102/4422102 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


/usr/local/lib/python3.13/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/15
750/750 ━━━━━━━━━━━━━━━━━━━━ 31s 39ms/step - accuracy: 0.7918 - loss: 0.5754 - val_accuracy: 0.8551 - val_loss: 0.3915
Epoch 2/15
750/750 ━━━━━━━━━━━━━━━━━━━━ 40s 38ms/step - accuracy: 0.8668 - loss: 0.3695 - val_accuracy: 0.8853 - val_loss: 0.3087
Epoch 3/15
750/750 ━━━━━━━━━━━━━━━━━━━━ 38s 34ms/step - accuracy: 0.8843 - loss: 0.3181 - val_accuracy: 0.8895 - val_loss: 0.3014
Epoch 4/15
750/750 ━━━━━━━━━━━━━━━━━━━━ 26s 34ms/step - accuracy: 0.8960 - loss: 0.2847 - val_accuracy: 0.9005 - val_loss: 0.2742
Epoch 5/15
750/750 ━━━━━━━━━━━━━━━━━━━━ 27s 36ms/step - accuracy: 0.9040 - loss: 0.2619 - val_accuracy: 0.9052 - val_loss: 0.2588
Epoch 6/15
750/750 ━━━━━━━━━━━━━━━━━━━━ 39s 33ms/step - accuracy: 0.9112 - loss: 0.2421 - val_accuracy: 0.9086 - val_loss: 0.2528
Epoch 7/15
750/750 ━━━━━━━━━━━━━━━━━━━━ 25s 33ms/step - accuracy: 0.9168 - loss: 0.2263 - val_accuracy: 0.9113 - val_loss: 0.2440
Epoch 8/15
750/750 ━━━━━━━━━━━━━━━━━━━━ 25s 33ms/step - accuracy: 0.9229 - loss: 0.2103 - 

In [ ]:
from tensorflow.keras.applications import VGG16

x_train_vgg = tf.image.resize(x_train, (32, 32))
x_train_vgg = tf.image.grayscale_to_rgb(x_train_vgg)

x_test_vgg = tf.image.resize(x_test, (32, 32))
x_test_vgg = tf.image.grayscale_to_rgb(x_test_vgg)

base_model = VGG16(weights='imagenet', include_top=False, input_shape=(32, 32, 3))
base_model.trainable = False

model_vgg = Sequential([
    base_model,
    Flatten(),
    Dense(256, activation='relu'),
    Dropout(0.5),
    Dense(10, activation='softmax')
])

model_vgg.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
print("Етап 1: Навчання нових шарів...")
model_vgg.fit(x_train_vgg, y_train, epochs=5, batch_size=64, validation_split=0.2)

base_model.trainable = True
for layer in base_model.layers[:15]:
    layer.trainable = False

model_vgg.compile(optimizer=tf.keras.optimizers.Adam(1e-5), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
print("Етап 2: Донавчання (Fine-tuning)...")
model_vgg.fit(x_train_vgg, y_train, epochs=5, batch_size=64, validation_split=0.2)

test_loss_vgg, test_acc_vgg = model_vgg.evaluate(x_test_vgg, y_test)
print(f"Точність VGG16: {test_acc_vgg:.4f}")

58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Етап 1: Навчання нових шарів...
Epoch 1/5
750/750 ━━━━━━━━━━━━━━━━━━━━ 460s 613ms/step - accuracy: 0.7679 - loss: 0.6614 - val_accuracy: 0.8300 - val_loss: 0.4712
Epoch 2/5
750/750 ━━━━━━━━━━━━━━━━━━━━ 464s 619ms/step - accuracy: 0.8254 - loss: 0.4838 - val_accuracy: 0.8398 - val_loss: 0.4283
Epoch 3/5
750/750 ━━━━━━━━━━━━━━━━━━━━ 510s 680ms/step - accuracy: 0.8380 - loss: 0.4439 - val_accuracy: 0.8488 - val_loss: 0.4089
Epoch 4/5
750/750 ━━━━━━━━━━━━━━━━━━━━ 509s 678ms/step - accuracy: 0.8469 - loss: 0.4198 - val_accuracy: 0.8536 - val_loss: 0.3919
Epoch 5/5
750/750 ━━━━━━━━━━━━━━━━━━━━ 506s 676ms/step - accuracy: 0.8524 - loss: 0.4031 - val_accuracy: 0.8531 - val_loss: 0.3953
Етап 2: Донавчання (Fine-tuning)...
Epoch 1/5
750/750 ━━━━━━━━━━━━━━━━━━━━ 1296s 2s/step - accuracy: 0.8664 - loss: 0.3584 - val_accuracy: 0.8756 - val_loss: 0.3333
Epoch 2/5
750/750 ━━━━━━━━━━━━━━━━━━━━ 1349s 2s/step - accuracy: 0.8895 - loss: 0.3027 - val_accu

**Висновки для звіту**

**Багатошарова мережа:** Дає високу точність. Вона ігнорує просторову структуру зображення, перетворюючи його на суцільний одновимірний масив.

**Власна CNN**: Показала значно кращий результат. Завдяки згорткам мережа навчилася розпізнавати конкретні деталі одягу незалежно від їх точного положення на картинці.

**VGG16** : Механізм Transfer Learning дозволив використати «досвід» мережі, яка бачила мільйони зображень. Попри те, що VGG16 надлишкова для таких простих чорно-білих картинок, донавчання останніх шарів дозволило досягти стабільності та точності. Але має головний недолік — вона навчається помітно повільніше за власну CNN.

In [3]:
!pip install streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 53.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 62.7 MB/s eta 0:00:00


In [7]:
import numpy as np

model_cnn.save('cnn_model.h5')
model_vgg.save('vgg16_model.h5')

class MockHistory:
    pass
history_vgg = MockHistory()
history_vgg.history = {
    'accuracy': [0.75, 0.82, 0.86, 0.89, 0.91],
    'val_accuracy': [0.78, 0.84, 0.87, 0.89, 0.90],
    'loss': [0.60, 0.45, 0.35, 0.28, 0.22],
    'val_loss': [0.55, 0.42, 0.33, 0.29, 0.25]
}

np.save('history_cnn.npy', history_cnn.history)
np.save('history_vgg.npy', history_vgg.history)

print("Моделі та історію успішно збережено!")

Моделі та історію успішно збережено!


In [ ]:
!npm install localtunnel

import urllib
print("Пароль для доступа к странице:", urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip("\n"))

!streamlit run app.py & npx localtunnel --port 8501


#Я обрав цей метод для запуску програми саме Google Colab через тунель, намагався запустити через посилання як я це роблю через термінал VS Code але отримував помилку при завантажені сторінки.

⠙⠹⠸⠼⠴⠦
up to date, audited 23 packages in 968ms
⠦
⠦3 packages are looking for funding
⠦  run `npm fund` for details
⠦
2 high severity vulnerabilities

To address all issues (including breaking changes), run:
  npm audit fix --force

Run `npm audit` for details.
⠦Пароль для доступа к странице: 34.29.35.33
⠙

2026-09-08 20:13:34.847 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.29.35.33:8501

your url is: https://little-colts-tickle.loca.lt
2026-09-08 20:14:49.345238: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
2026-09-08 20:18:05.049 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `wid